# 03 Modeling: Four-Algorithm Temporal Graph Comparison

This notebook runs the final four-algorithm comparison for the bonus requirement.

Algorithms:
1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM

Final feature set:
- Raw transaction features
- Historical graph features
- Rolling temporal graph features

Model selection metric:
- Validation PR-AUC, because AML is a highly imbalanced ranking problem.

In [1]:
from pathlib import Path
import sys

# Works whether the notebook is launched from repository root or from notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from src.data.load_data import chronological_split
from src.features.tabular_features import make_model_matrix

PROJECT_ROOT: /home/acer/dev/aml-graph-dm


## 1. Configuration

In [2]:
DATA_PATH = PROCESSED_DIR / "hi_small_features.parquet"
NEG_PER_POS = 100
INCLUDE_GRAPH = True

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed data not found: {DATA_PATH}\n"
        "Run 02_feature_engineering_completed.ipynb first."
    )

print("DATA_PATH:", DATA_PATH)
print("NEG_PER_POS:", NEG_PER_POS)
print("INCLUDE_GRAPH:", INCLUDE_GRAPH)

DATA_PATH: /home/acer/dev/aml-graph-dm/data/processed/hi_small_features.parquet
NEG_PER_POS: 100
INCLUDE_GRAPH: True


## 2. Helper functions

In [3]:
def sample_training_data(X, y, neg_per_pos: int = 100, random_state: int = 42):
    """Keep all positive samples and sample negative samples for faster imbalanced training."""
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    if n_pos == 0:
        raise ValueError("No positive samples in training data.")

    n_neg = min(len(neg_idx), n_pos * neg_per_pos)
    sampled_neg_idx = neg_idx.to_series().sample(n=n_neg, random_state=random_state).index
    sampled_idx = pos_idx.union(sampled_neg_idx)

    return X.loc[sampled_idx], y.loc[sampled_idx]


def evaluate_model(model, X, y, threshold: float = 0.5) -> dict:
    """Evaluate a fitted binary classifier using AML-relevant metrics."""
    scores = model.predict_proba(X)[:, 1]
    pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    return {
        "pr_auc": float(average_precision_score(y, scores)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "alert_count": int(pred.sum()),
    }


def get_fast_models(scale_pos_weight: float) -> dict:
    """Return the four algorithms used in the final comparison."""
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler(with_mean=False)),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        solver="saga",
                        n_jobs=-1,
                        random_state=42,
                    ),
                ),
            ]
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=100,
            max_depth=14,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            max_samples=0.8,
            n_jobs=-1,
            random_state=42,
        ),
        "xgboost": XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="aucpr",
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        ),
        "lightgbm": LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=30,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1,
        ),
    }

## 3. Load data and split chronologically

In [4]:
df = pd.read_parquet(DATA_PATH)

train_df, val_df, test_df = chronological_split(df, train_size=0.60, val_size=0.20)

print("Data shape:", df.shape)
print("\nSplit shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nSplit label counts:")
print("Train:")
display(train_df["is_laundering"].value_counts())
print("Validation:")
display(val_df["is_laundering"].value_counts())
print("Test:")
display(test_df["is_laundering"].value_counts())

Data shape: (2000000, 63)

Split shapes:
Train: (1200000, 63)
Validation: (400000, 63)
Test: (400000, 63)

Split label counts:
Train:


is_laundering
0    1199639
1        361
Name: count, dtype: int64

Validation:


is_laundering
0    399773
1       227
Name: count, dtype: int64

Test:


is_laundering
0    399090
1       910
Name: count, dtype: int64

## 4. Build model matrices

In [5]:
X_train, y_train = make_model_matrix(train_df, include_graph_features=INCLUDE_GRAPH)
X_val, y_val = make_model_matrix(val_df, include_graph_features=INCLUDE_GRAPH)
X_test, y_test = make_model_matrix(test_df, include_graph_features=INCLUDE_GRAPH)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (1200000, 92)
X_val: (400000, 92)
X_test: (400000, 92)


## 5. Negative sampling for imbalanced training

In [6]:
X_sample, y_sample = sample_training_data(
    X_train,
    y_train,
    neg_per_pos=NEG_PER_POS,
    random_state=42,
)

pos = max(int(y_sample.sum()), 1)
neg = max(len(y_sample) - pos, 1)
scale_pos_weight = neg / pos

print("Sampled training shape:", X_sample.shape)
print("Sampled positives:", pos)
print("Sampled negatives:", neg)
print("scale_pos_weight:", scale_pos_weight)

Sampled training shape: (36461, 92)
Sampled positives: 361
Sampled negatives: 36100
scale_pos_weight: 100.0


## 6. Train and evaluate four algorithms

In [7]:
models = get_fast_models(scale_pos_weight=scale_pos_weight)

rows = []
fitted_models = {}

for model_name, model in models.items():
    print(f"\n=== Training {model_name} ===")
    model.fit(X_sample, y_sample)

    val_metrics = evaluate_model(model, X_val, y_val, threshold=0.5)
    test_metrics = evaluate_model(model, X_test, y_test, threshold=0.5)

    row = {
        "model": model_name,
        "feature_set": "raw_plus_temporal_graph" if INCLUDE_GRAPH else "raw",
        "neg_per_pos": NEG_PER_POS,
        "feature_count": X_train.shape[1],
        "train_sample_rows": len(X_sample),
        "train_sample_pos": int(y_sample.sum()),
        "train_sample_neg": int(len(y_sample) - y_sample.sum()),
        "val_pr_auc": val_metrics["pr_auc"],
        "val_f1": val_metrics["f1"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_f1": test_metrics["f1"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_tp": test_metrics["tp"],
        "test_fp": test_metrics["fp"],
        "test_tn": test_metrics["tn"],
        "test_fn": test_metrics["fn"],
        "test_alert_count": test_metrics["alert_count"],
    }

    rows.append(row)
    fitted_models[model_name] = model
    display(pd.DataFrame([row]))

comparison = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False)
display(comparison)

comparison_path = TABLES_DIR / "four_model_comparison_temporal_graph.csv"
comparison_final_path = TABLES_DIR / "four_model_comparison_temporal_graph_final.csv"

comparison.to_csv(comparison_path, index=False)
comparison.to_csv(comparison_final_path, index=False)

print("Saved:", comparison_path)
print("Saved:", comparison_final_path)


=== Training logistic_regression ===


/home/acer/dev/aml-graph-dm/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/home/acer/dev/aml-graph-dm/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,feature_set,neg_per_pos,feature_count,train_sample_rows,train_sample_pos,train_sample_neg,val_pr_auc,val_f1,val_precision,val_recall,test_pr_auc,test_f1,test_precision,test_recall,test_tp,test_fp,test_tn,test_fn,test_alert_count
0,logistic_regression,raw_plus_temporal_graph,100,92,36461,361,36100,0.006419,0.006578,0.003301,0.942731,0.133309,0.027266,0.013825,0.981319,893,63699,335391,17,64592



=== Training random_forest ===


,model,feature_set,neg_per_pos,feature_count,train_sample_rows,train_sample_pos,train_sample_neg,val_pr_auc,val_f1,val_precision,val_recall,test_pr_auc,test_f1,test_precision,test_recall,test_tp,test_fp,test_tn,test_fn,test_alert_count
0,random_forest,raw_plus_temporal_graph,100,92,36461,361,36100,0.02396,0.024277,0.012399,0.577093,0.042708,0.092769,0.05057,0.56044,510,9575,389515,400,10085


...

,model,feature_set,neg_per_pos,feature_count,train_sample_rows,train_sample_pos,train_sample_neg,val_pr_auc,val_f1,val_precision,val_recall,test_pr_auc,test_f1,test_precision,test_recall,test_tp,test_fp,test_tn,test_fn,test_alert_count
0,xgboost,raw_plus_temporal_graph,100,92,36461,361,36100,0.032368,0.038728,0.020178,0.480176,0.053348,0.108758,0.063674,0.372527,339,4985,394105,571,5324



=== Training lightgbm ===
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 361, number of negative: 36100
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.065509 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10428
[LightGBM] [Info] Number of data points in the train set: 36461, number of used features: 89
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.009901 -> initscore=-4.605170
[LightGBM] [Info] Start training from score -4.605170


,model,feature_set,neg_per_pos,feature_count,train_sample_rows,train_sample_pos,train_sample_neg,val_pr_auc,val_f1,val_precision,val_recall,test_pr_auc,test_f1,test_precision,test_recall,test_tp,test_fp,test_tn,test_fn,test_alert_count
0,lightgbm,raw_plus_temporal_graph,100,92,36461,361,36100,0.035802,0.073724,0.041292,0.343612,0.081038,0.154743,0.109683,0.262637,239,1940,397150,671,2179


,model,feature_set,neg_per_pos,feature_count,train_sample_rows,train_sample_pos,train_sample_neg,val_pr_auc,val_f1,val_precision,val_recall,test_pr_auc,test_f1,test_precision,test_recall,test_tp,test_fp,test_tn,test_fn,test_alert_count
3,lightgbm,raw_plus_temporal_graph,100,92,36461,361,36100,0.035802,0.073724,0.041292,0.343612,0.081038,0.154743,0.109683,0.262637,239,1940,397150,671,2179
2,xgboost,raw_plus_temporal_graph,100,92,36461,361,36100,0.032368,0.038728,0.020178,0.480176,0.053348,0.108758,0.063674,0.372527,339,4985,394105,571,5324
1,random_forest,raw_plus_temporal_graph,100,92,36461,361,36100,0.023960,0.024277,0.012399,0.577093,0.042708,0.092769,0.050570,0.560440,510,9575,389515,400,10085
0,logistic_regression,raw_plus_temporal_graph,100,92,36461,361,36100,0.006419,0.006578,0.003301,0.942731,0.133309,0.027266,0.013825,0.981319,893,63699,335391,17,64592


Saved: /home/acer/dev/aml-graph-dm/reports/tables/four_model_comparison_temporal_graph.csv
Saved: /home/acer/dev/aml-graph-dm/reports/tables/four_model_comparison_temporal_graph_final.csv


## 7. Save best model bundle

In [8]:
best_row = comparison.iloc[0]
best_model_name = best_row["model"]
best_model = fitted_models[best_model_name]

bundle = {
    "model": best_model,
    "model_name": best_model_name,
    "feature_set": "raw_plus_graph" if INCLUDE_GRAPH else "raw",
    "feature_columns": list(X_train.columns),
    "threshold": 0.5,
    "selection_metric": "val_pr_auc",
    "val_pr_auc": float(best_row["val_pr_auc"]),
    "test_pr_auc": float(best_row["test_pr_auc"]),
    "training_note": (
        "Best model selected from Logistic Regression, Random Forest, XGBoost, "
        "and LightGBM using validation PR-AUC. Training used chronological split "
        "and negative undersampling."
    ),
}

model_path = MODELS_DIR / "best_model_bundle.joblib"
joblib.dump(bundle, model_path)

print("Saved best model bundle:", model_path)
print("Best model:", best_model_name)
print("Validation PR-AUC:", best_row["val_pr_auc"])
print("Test PR-AUC:", best_row["test_pr_auc"])

Saved best model bundle: /home/acer/dev/aml-graph-dm/models/best_model_bundle.joblib
Best model: lightgbm
Validation PR-AUC: 0.03580155240879152
Test PR-AUC: 0.08103786689958767


## Modeling notes for the report

Use these points in BAB III and BAB IV:
- Four algorithms were compared on the same final temporal graph feature set.
- Model selection used validation PR-AUC, not test PR-AUC.
- Negative sampling was used only on the training split.
- LightGBM is selected if it has the highest validation PR-AUC and manageable alert volume.